# EDA — NYC Yellow Taxi (couche Silver)

Analyse exploratoire des données de courses de taxi new-yorkaises, à partir de la couche **Silver** (données nettoyées), exportée localement depuis HDFS.

**Prérequis (une seule fois) :**
```bash
docker exec -u root namenode hdfs dfs -get /data/silver/taxi /tmp/silver_taxi
docker cp namenode:/tmp/silver_taxi ./data/silver_local
```

Ce notebook tourne ensuite **entièrement en local avec pandas** — pas besoin de Spark ni de Docker pour l'EDA elle-même (Spark reste utilisé uniquement pour le traitement Bronze → Silver → Gold en amont).

In [1]:
import glob
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

SILVER_LOCAL_PATH = "../data/silver_local"  # adapte le chemin si besoin

In [2]:
parquet_files = glob.glob(f"{SILVER_LOCAL_PATH}/**/*.parquet", recursive=True)
print(f"{len(parquet_files)} fichier(s) Parquet trouvé(s).")

df = pd.concat([pd.read_parquet(f) for f in parquet_files], ignore_index=True)
print(f"Nombre de lignes : {len(df):,}")
print(f"Nombre de colonnes : {len(df.columns)}")
df.dtypes

653 fichier(s) Parquet trouvé(s).
Nombre de lignes : 42,402,283
Nombre de colonnes : 27


VendorID                          int32
pickup_datetime          datetime64[ns]
dropoff_datetime         datetime64[ns]
passenger_count                   int32
trip_distance                   float64
RatecodeID                        int64
store_and_fwd_flag               object
PULocationID                      int32
DOLocationID                      int32
payment_type                      int64
fare_amount                     float64
extra                           float64
mta_tax                         float64
tip_amount                      float64
tolls_amount                    float64
improvement_surcharge           float64
total_amount                    float64
congestion_surcharge            float64
Airport_fee                     float64
cbd_congestion_fee              float64
pickup_hour                       int32
pickup_dayofweek                  int32
is_weekend                         bool
trip_duration_minutes           float64
price_per_mile                  float64


## 1. Statistiques descriptives

In [3]:
num_cols = [
    "trip_distance", "fare_amount", "tip_amount", "tolls_amount",
    "total_amount", "passenger_count", "trip_duration_minutes", "price_per_mile",
]
df[num_cols].describe()

MemoryError: Unable to allocate 4.11 GiB for an array with shape (13, 42402283) and data type float64

In [4]:
print(df.shape)
print(df.info(memory_usage="deep"))

(42402283, 27)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42402283 entries, 0 to 42402282
Data columns (total 27 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   VendorID               int32         
 1   pickup_datetime        datetime64[ns]
 2   dropoff_datetime       datetime64[ns]
 3   passenger_count        int32         
 4   trip_distance          float64       
 5   RatecodeID             int64         
 6   store_and_fwd_flag     object        
 7   PULocationID           int32         
 8   DOLocationID           int32         
 9   payment_type           int64         
 10  fare_amount            float64       
 11  extra                  float64       
 12  mta_tax                float64       
 13  tip_amount             float64       
 14  tolls_amount           float64       
 15  improvement_surcharge  float64       
 16  total_amount           float64       
 17  congestion_surcharge   float64       
 18  Airpo

## 2. Valeurs manquantes

In [ ]:
missing = df.isnull().sum()
missing[missing > 0].sort_values(ascending=False)

## 3. Distribution des prix

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df["total_amount"], bins=60, kde=True, ax=axes[0])
axes[0].set_title("Distribution du prix total (total_amount)")
axes[0].set_xlabel("Prix ($)")

sns.boxplot(x=df["total_amount"], ax=axes[1])
axes[1].set_title("Boxplot du prix total")
axes[1].set_xlabel("Prix ($)")

plt.tight_layout()
plt.show()

## 4. Distribution des distances

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df["trip_distance"], bins=60, kde=True, ax=axes[0])
axes[0].set_title("Distribution de la distance (miles)")
axes[0].set_xlabel("Distance (miles)")

sns.boxplot(x=df["trip_distance"], ax=axes[1])
axes[1].set_title("Boxplot de la distance")
axes[1].set_xlabel("Distance (miles)")

plt.tight_layout()
plt.show()

## 5. Répartition des courses selon l'heure et le jour

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.countplot(x="pickup_hour", data=df, color="steelblue", ax=axes[0])
axes[0].set_title("Nombre de courses par heure de la journée")
axes[0].set_xlabel("Heure")

day_labels = {1: "Dim", 2: "Lun", 3: "Mar", 4: "Mer", 5: "Jeu", 6: "Ven", 7: "Sam"}
df["day_label"] = df["pickup_dayofweek"].map(day_labels)
order = ["Lun", "Mar", "Mer", "Jeu", "Ven", "Sam", "Dim"]
sns.countplot(x="day_label", data=df, order=order, color="darkorange", ax=axes[1])
axes[1].set_title("Nombre de courses par jour de la semaine")
axes[1].set_xlabel("Jour")

plt.tight_layout()
plt.show()

## 6. Prix moyen selon l'heure

In [ ]:
hourly_avg = df.groupby("pickup_hour")["total_amount"].mean().reset_index()

plt.figure(figsize=(10, 5))
sns.lineplot(x="pickup_hour", y="total_amount", data=hourly_avg, marker="o")
plt.title("Prix moyen par heure de la journée")
plt.xlabel("Heure")
plt.ylabel("Prix moyen ($)")
plt.show()

## 7. Top zones de départ

In [ ]:
top_pickup = df["PULocationID"].value_counts().head(15).reset_index()
top_pickup.columns = ["PULocationID", "count"]

plt.figure(figsize=(10, 6))
sns.barplot(x="count", y="PULocationID", data=top_pickup, orient="h", color="mediumseagreen", order=top_pickup["PULocationID"])
plt.title("Top 15 zones de départ (par ID de zone TLC)")
plt.xlabel("Nombre de courses")
plt.ylabel("PULocationID")
plt.show()

## 8. Corrélations

In [ ]:
corr_cols = ["trip_distance", "trip_duration_minutes", "passenger_count", "tip_amount", "total_amount"]
corr_matrix = df[corr_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", vmin=-1, vmax=1, fmt=".2f")
plt.title("Heatmap des corrélations")
plt.show()

## 9. Synthèse

À compléter après lecture des graphiques ci-dessus :
- Quels facteurs semblent le plus corrélés au prix ?
- Y a-t-il des patterns horaires ou hebdomadaires clairs ?
- Quelles zones concentrent le plus de départs ?

Ces observations orienteront le choix des variables explicatives pour le modèle de Machine Learning (`spark/ml_training.py`).